### Universidade Federal da Bahia
### Escola Politécnica
### Programa de Pós-Graduação em Engenharia Industrial
### Discente: Ghabriel Anton Gomes de Sá
### Orientadores: Marcelo Embiruçu e Cristiano Fontes

<center><h1>UMA ABORDAGEM COMBINANDO UMA ESTIMATIVA DE PESOS INICIAIS BASEADA EM LINEARIZAÇÃO E ALGORITMO CONSTRUTIVISTA PARA REDES NEURAIS DE PROPAGAÇÃO DIRETA COM UMA ÚNICA CAMADA OCULTA APLICADAS À IDENTIFIÇÃO DE MODELOS COM MÚLTIPLAS ENTRADAS E ÚNICA SAÍDA EM PROBLEMAS DE REGRESSÃO</center></h1>

#### <p style='text-align: justify;'>O Método para a obtenção dos pesos iniciais propõe que: (i) seja realizada a linearização, via série de Taylor, do modelo não-linear a ser utilizado para a modelagem do conjunto de dados (o modelo consiste em uma rede neural com p entradas e um neurônio na camada de saída), onde o ponto de equilíbrio para a linearização é a média das suas respectivas variáveis (x,y); (ii) seja realizada a Regressão Linear Múltipla, via Método dos Mínimos Quadrados, do conjunto de dados a ser modelado; (iii) através da comparação matemática entre (i) e (ii), obtenha-se os pesos que conectam os neurônios de entrada ao primeiro neurônio oculto da rede.
#### <p style='text-align: justify;'>Em seguida, a rede continua a aumentar o número de neurônios ocultos a partir de uma abordagem construtivista. Toda a rede é treinada a cada adição de um novo neurônio oculto, sendo que os pesos (e bias) obtidos no final de um treinamento são utilizados como chutes iniciais para o treinamento subsequente. Os pesos iniciais do neurônio oculto a ser adicionado são obtidos a partir do método de Bengio e Glorot (2010), conhecido como método de Xavier; biases são inicializados com valor nulo. Comparou-se o método proposto (WILCA-R) com outros dois métodos (RIXM e ELM).

## Importação de pacotes

In [ ]:
#Manipulação de dados:
import pandas as pd
from sklearn import preprocessing

#K-fold cross validation:
from sklearn.model_selection import KFold

#Resolução de sistema de eq. não-lineares:
import statsmodels.api as sm
from scipy.optimize import fsolve
from scipy.optimize import minimize

#Álgebra linear:
import numpy as np
import math
from scipy.linalg import pinv2 #Usado no ELM

#Cálculo do R2:
from sklearn.metrics import r2_score

#Contagem de tempo:
import time

#Criação e plotagem de gráficos:
import matplotlib.pyplot as plt
%matplotlib inline

## Importação do banco de dados
<p style='text-align: justify;'>Para que a importação do banco de dados seja feita da maneira correta, é necessário que o banco de dados a ser importado esteja de acordo com o padrão apresentado a seguir e também que ele esteja localizado na mesma pasta que esse código. A princípio, esse código está implementado em um arquivo .ipynb.</p>

O banco de dados deve apresentar, obrigatoriamente, as seguintes características:

-Deve ser um arquivo do tipo .csv;

-Os dados devem estar delimitados por ponto e vírgula (;), ou seja, devem estar presentes em diferentes células da planilha Excel;

-As variáveis de entradas são todas as colunas, exceto a última (que trata-se da variável de saída);

-Não deveve possuir cabeçalho (linha com nome dos atributos).

In [ ]:
#Pedindo informações do conjunto de dados (nome do conjunto e se ele possui cabeçalho ou não):
filename = input("Digite o nome do arquivo .csv que contém o banco de dados:")

#Abrindo conjunto de dados:
data = pd.read_csv(filename+'.csv', header = None, sep = ';')

## Pré-processamento de dados

#### • Obtendo primeiras informações sobre o conjunto de dados:

In [ ]:
#Mostrando cinco primeiras linhas de dados:
data.head()

In [ ]:
#Mostrando informações sobre os atributos:
data.info()

In [ ]:
#Descrição estatística dos dados:
data.describe()

#### • Lidando com os dados faltantes:

In [ ]:
#Verificando a existência de dados faltantes:
data.isna().sum()

In [ ]:
#Deletando dados faltantes:
data.dropna(inplace=True)

In [ ]:
#Missing values também podem estar representadas através do número 0. Por conta disso, deve-se verificar se os zeros são missing values ou se são
#inerentes ao atributo:
num_zeros = (data == 0).sum()
print('Quantidade de valores nulos:\n',num_zeros)

#### • Reduzindo a escala dos dados:
Essa etapa garante que os dados estejam entre 0 e 1.

In [ ]:
#Cria uma array a partir dos dados:
z = data.values

#Atribuindo nova escala os dados:
min_max_scaler = preprocessing.MinMaxScaler()
data_normalized = min_max_scaler.fit_transform(z)

In [ ]:
print("Dados normalizados:\n",data_normalized)

#### • Delimitação de entrada e saída:

In [ ]:
#Delimitando:
inputs = data_normalized[:,:-1]
targets = data_normalized[:,-1]

#Atribuindo valor ao número de atributos (número de colunas dos dados de entrada):
n_inputs = inputs.shape[1]

In [ ]:
#Exibindo valores em tela:
print("Inputs:\n",inputs)
print("Outputs:\n",targets)
print("Nº de atributos:\n",n_inputs)

## Criação dos Modelos
<p style='text-align: justify;'>Os modelos são do tipo SFNN e treinados de acordo com o algoritmo de retropropagação (ARP); exceto o Método 3, que utiliza o ELM.</p>

## Definindo função para cálculo de MSE
MSE é a função de perda, utilizado para a otimização do ARP.

In [ ]:
def mse(actual, pred): 
    actual, pred = np.array(actual), np.array(pred)
    return np.square(np.subtract(actual,pred)).mean() 

## MÉTODO 1 COM RESTRIÇÕES DE SINAIS DE GANHOS
Modelo SFNN, treinado via ARP, considerando os pesos iniciais (e bias) do primeiro neurônio oculto obtidos a partir de Linearização e Análise via Série de Taylor e pesos dos demais neurônios ocultos, bem como do neurônio de saída obtidos a partir do método de Xavier, preservando os pesos (e bias) já aprendidos e utilizando-os como estimativas iniciais para o próximo treinamento. Novos pesos são inicializados via método de Xavier e novos biases são inicializados com valor nulo.

### Aplicação de Regressão Linear no conjunto de dados
<p style='text-align: justify;'>A regressão é feita utilizando o módulo statsmodels.api. O método utilizado para a regressão é o Método dos Mínimos Quadrados.</p>

In [ ]:
inputs_cte = sm.add_constant(inputs, prepend=True) #Adiciona a coluna das constantes
res = sm.OLS(targets,inputs_cte).fit() #Cria e ajusta o modelo
print (res.summary()) #Imprime os resultados

### Resolução do sistema de equações proposto para obter os parâmetros iniciais da rede
<p style='text-align: justify;'>A resolução é feita utilizando o módulo scipy.optimize.fsolve. O módulo retorna as raízes das equações (não lineares) definidas por f(x)= 0, dada uma estimativa inicial.</p>

In [ ]:
#Definindo constantes:
x_M = []
for i in range(n_inputs):
    x_M.append(np.mean(inputs[:,i]))
k = []
for i in range(n_inputs):
    k.append(res.params[i+1])
b = res.params[0]

In [ ]:
#Definindo chutes iniciais para beta e wi como valores aleatórios entre 0 e 1:
x0 = np.random.rand(n_inputs+1,1)

In [ ]:
#Definindo função sigmoide e sua derivada:
def sigmoid(x):
    return 1 / (1 + np.exp(-x))

def dsigmoid(x):
    z = sigmoid(x)
    return z * (1 - z)

In [ ]:
#Inicializando variáveis wi e beta:
w = [0 for i in range(n_inputs)]
beta = 0

In [ ]:
#Definindo sistema de equações a ser resolvido:
def f(x):
    w,beta = x[:-1],x[-1]
    f = [sigmoid(np.dot(w,x_M)+beta)-(sigmoid(np.dot(w,x_M)+beta)-(sigmoid(np.dot(w,x_M)+beta))**2)*np.dot(w,x_M)-b]
    for i in range(n_inputs):
        f.append((sigmoid(np.dot(w,x_M)+beta)-(sigmoid(np.dot(w,x_M)+beta))**2)*w[i]-k[i])
    return f

In [ ]:
#Resolvendo o sistema e imprimindo resultados (pesos e bias iniciais):
result = fsolve(f, x0)
print(result)

### Executando Método 1:

In [ ]:
#Inicializando contagem do tempo:
ini=time.time()

#Abrindo listas para RMSE e R2 (treino e teste):

rmse_train_opt = []
rmse_test_opt = []
r2_train_opt = []
r2_test_opt = []

#Inicializando loop de experimentos:
l=1
kf = KFold(5, True, 1)
for train, test in kf.split(inputs, targets):
    np.random.seed(l)
    print("Validação cruzada",l,)
    #Abrindo listas para RMSE, MSE e R2 (treino e teste):
    mse_train = []
    mse_test = []
    rmse_train = []
    rmse_test = []
    r2_train = []
    r2_test = []

    #Sinalização, em tela, de início de rede com 1 neurônio oculto:
    print("Iniciando rede com 1 neurônio oculto:")

    #Definindo entrada e saída de treino:
    x = inputs[train].T
    y = targets[train].T

   # Definindo pesos iniciais da rede
    W1_in = np.random.uniform(-math.sqrt(6/(x.shape[0] + 1)), math.sqrt(6/(x.shape[0] + 1)), (1, x.shape[0]))
    W1_in[0] = result[:-1]
    b1_in = np.zeros((1, 1))
    b1_in[0] = result[-1]
    W2_in = np.random.uniform(-math.sqrt(6/(1 + 1)), math.sqrt(6/(1 + 1)), (1, 1))
    b2_in = np.zeros((1, 1))

    # Definindo vetor de pesos iniciais da função de optimização
    W0 = np.concatenate((W1_in.flatten(), b1_in.flatten(), W2_in.flatten(), b2_in.flatten()))

    # Definindo função objetivo (loss function):
    def objetivo(W):
        W1 = W[0:x.shape[0]].reshape(1,x.shape[0])
        b1 = W[x.shape[0]:x.shape[0]+1].reshape(1,1)
        W2 = W[x.shape[0]+1:-1].reshape(1,1)
        b2 = W[-1].reshape(1,1)
        Z1 = W1.dot(x) + b1
        A1 = sigmoid(Z1)
        Z2 = W2.dot(A1) + b2
        A2 = sigmoid(Z2)[0]
        loss = mse(y, A2)
        return math.sqrt(loss)

    def restricao1(W):
        W1 = W[0:x.shape[0]].reshape(1,x.shape[0])
        b1 = W[x.shape[0]:x.shape[0]+1].reshape(1,1)
        W2 = W[x.shape[0]+1:-1].reshape(1,1)
        b2 = W[-1].reshape(1,1)

        # Definindo valor base de x (tomando primeira entrada do conjunto de treino como referência)
        x_base = []
        for k in range(x.shape[0]):
            x_base.append(x[k][0])
        x_base = np.asarray(x_base)
        x_base = x_base.reshape(x.shape[0],1)
        # Definindo valor base de y
        Z1 = W1.dot(x_base) + b1
        A1 = sigmoid(Z1)
        Z2 = W2.dot(A1) + b2
        A2 = sigmoid(Z2)
        y_base = A2[0]

        # Registrando ganho referente à variavel 1 
        x_var = x_base.copy()
        x_var[0] = x_var[0]+0.1
        Z1 = W1.dot(x_var) + b1
        A1 = sigmoid(Z1)
        Z2 = W2.dot(A1) + b2
        A2 = sigmoid(Z2)
        new_y = A2[0]
        k1 = (new_y - y_base) / (x_var[0] - x_base[0])
        return -k1[0] - 0.0000000001

    def restricao2(W):
        W1 = W[0:x.shape[0]].reshape(1,x.shape[0])
        b1 = W[x.shape[0]:x.shape[0]+1].reshape(1,1)
        W2 = W[x.shape[0]+1:-1].reshape(1,1)
        b2 = W[-1].reshape(1,1)

        # Definindo valor base de x (tomando primeira entrada do conjunto de treino como referência)
        x_base = []
        for k in range(x.shape[0]):
            x_base.append(x[k][0])
        x_base = np.asarray(x_base)
        x_base = x_base.reshape(x.shape[0],1)
        # Definindo valor base de y
        Z1 = W1.dot(x_base) + b1
        A1 = sigmoid(Z1)
        Z2 = W2.dot(A1) + b2
        A2 = sigmoid(Z2)
        y_base = A2[0]

        # Registrando ganho referente à variavel 2
        x_var = x_base.copy()
        x_var[1] = x_var[1]+0.1
        Z1 = W1.dot(x_var) + b1
        A1 = sigmoid(Z1)
        Z2 = W2.dot(A1) + b2
        A2 = sigmoid(Z2)
        new_y = A2[0]
        k2 = (new_y - y_base) / (x_var[1] - x_base[1])
        return k2[0] - 0.0000000001

    def restricao3(W):
        W1 = W[0:x.shape[0]].reshape(1,x.shape[0])
        b1 = W[x.shape[0]:x.shape[0]+1].reshape(1,1)
        W2 = W[x.shape[0]+1:-1].reshape(1,1)
        b2 = W[-1].reshape(1,1)

        # Definindo valor base de x (tomando primeira entrada do conjunto de treino como referência)
        x_base = []
        for k in range(x.shape[0]):
            x_base.append(x[k][0])
        x_base = np.asarray(x_base)
        x_base = x_base.reshape(x.shape[0],1)
        # Definindo valor base de y
        Z1 = W1.dot(x_base) + b1
        A1 = sigmoid(Z1)
        Z2 = W2.dot(A1) + b2
        A2 = sigmoid(Z2)
        y_base = A2[0]

       # Registrando ganho referente à variavel 3
        x_var = x_base.copy()
        x_var[2] = x_var[2]+0.1
        Z1 = W1.dot(x_var) + b1
        A1 = sigmoid(Z1)
        Z2 = W2.dot(A1) + b2
        A2 = sigmoid(Z2)
        new_y = A2[0]
        k3 = (new_y - y_base) / (x_var[2] - x_base[2])
        return k3[0] - 0.0000000001

    def restricao4(W):
        W1 = W[0:x.shape[0]].reshape(1,x.shape[0])
        b1 = W[x.shape[0]:x.shape[0]+1].reshape(1,1)
        W2 = W[x.shape[0]+1:-1].reshape(1,1)
        b2 = W[-1].reshape(1,1)

        # Definindo valor base de x (tomando primeira entrada do conjunto de treino como referência)
        x_base = []
        for k in range(x.shape[0]):
            x_base.append(x[k][0])
        x_base = np.asarray(x_base)
        x_base = x_base.reshape(x.shape[0],1)
        # Definindo valor base de y
        Z1 = W1.dot(x_base) + b1
        A1 = sigmoid(Z1)
        Z2 = W2.dot(A1) + b2
        A2 = sigmoid(Z2)
        y_base = A2[0]

        # Registrando ganho referente à variavel 4
        x_var = x_base.copy()
        x_var[3] = x_var[3]+0.1
        Z1 = W1.dot(x_var) + b1
        A1 = sigmoid(Z1)
        Z2 = W2.dot(A1) + b2
        A2 = sigmoid(Z2)
        new_y = A2[0]
        k4 = (new_y - y_base) / (x_var[3] - x_base[3])
        return k4[0] - 0.0000000001

    def restricao5(W):
        W1 = W[0:x.shape[0]].reshape(1,x.shape[0])
        b1 = W[x.shape[0]:x.shape[0]+1].reshape(1,1)
        W2 = W[x.shape[0]+1:-1].reshape(1,1)
        b2 = W[-1].reshape(1,1)

        # Definindo valor base de x (tomando primeira entrada do conjunto de treino como referência)
        x_base = []
        for k in range(x.shape[0]):
            x_base.append(x[k][0])
        x_base = np.asarray(x_base)
        x_base = x_base.reshape(x.shape[0],1)
        # Definindo valor base de y
        Z1 = W1.dot(x_base) + b1
        A1 = sigmoid(Z1)
        Z2 = W2.dot(A1) + b2
        A2 = sigmoid(Z2)
        y_base = A2[0]

        # Registrando ganho referente à variavel 5
        x_var = x_base.copy()
        x_var[4] = x_var[4]+0.1
        Z1 = W1.dot(x_var) + b1
        A1 = sigmoid(Z1)
        Z2 = W2.dot(A1) + b2
        A2 = sigmoid(Z2)
        new_y = A2[0]
        k5 = (new_y - y_base) / (x_var[4] - x_base[4])
        return k5[0] - 0.0000000001

    def restricao6(W):
        W1 = W[0:x.shape[0]].reshape(1,x.shape[0])
        b1 = W[x.shape[0]:x.shape[0]+1].reshape(1,1)
        W2 = W[x.shape[0]+1:-1].reshape(1,1)
        b2 = W[-1].reshape(1,1)

        # Definindo valor base de x (tomando primeira entrada do conjunto de treino como referência)
        x_base = []
        for k in range(x.shape[0]):
            x_base.append(x[k][0])
        x_base = np.asarray(x_base)
        x_base = x_base.reshape(x.shape[0],1)
        # Definindo valor base de y
        Z1 = W1.dot(x_base) + b1
        A1 = sigmoid(Z1)
        Z2 = W2.dot(A1) + b2
        A2 = sigmoid(Z2)
        y_base = A2[0]

        # Registrando ganho referente à variavel 6
        x_var = x_base.copy()
        x_var[5] = x_var[5]+0.1
        Z1 = W1.dot(x_var) + b1
        A1 = sigmoid(Z1)
        Z2 = W2.dot(A1) + b2
        A2 = sigmoid(Z2)
        new_y = A2[0]
        k6 = (new_y - y_base) / (x_var[5] - x_base[5])
        return k6[0] - 0.0000000001

    rest1 = {'type': 'ineq', 'fun': restricao1}
    rest2 = {'type': 'ineq', 'fun': restricao2}
    rest3 = {'type': 'ineq', 'fun': restricao3}
    rest4 = {'type': 'ineq', 'fun': restricao4}
    rest5 = {'type': 'ineq', 'fun': restricao5}
    rest6 = {'type': 'ineq', 'fun': restricao6}

    cons = ([rest1, rest2, rest3, rest4, rest5, rest6])
    
    # Implementando otimização
    solucao = minimize(objetivo, W0, method = 'SLSQP', constraints = cons)
    
    # Atribuindo pesos obtidos após à otimização
    W1 = solucao['x'][0:x.shape[0]*1].reshape(1,x.shape[0])
    b1 = solucao['x'][x.shape[0]:x.shape[0]+1].reshape(1,1)
    W2 = solucao['x'][x.shape[0]+1:-1].reshape(1,1)
    b2 = solucao['x'][-1].reshape(1,1)
    
    #MSE e RMSE ao fim do treino:
    Z1 = W1.dot(x) + b1
    A1 = sigmoid(Z1)
    Z2 = W2.dot(A1) + b2
    A2 = sigmoid(Z2)
    y_predict = A2[0]
    Modelo1_loss = mse(y, y_predict)
    mse_train.append(Modelo1_loss)
    rmse_train.append(np.sqrt(Modelo1_loss))

    #R2 de treino:
    r2_train.append(r2_score(y, y_predict))

    #Definindo entrada e saída de teste:
    x = inputs[test].T
    y = targets[test].T

    #Testando modelo e calculando MSE de teste:
    Z1 = W1.dot(x) + b1
    A1 = sigmoid(Z1)
    Z2 = W2.dot(A1) + b2
    A2 = sigmoid(Z2)
    y_predict = A2[0]
    Modelo1_loss = mse(y, y_predict)
    mse_test.append(Modelo1_loss)
    rmse_test.append(np.sqrt(Modelo1_loss))

    #R2 de teste:
    r2_test.append(r2_score(y, y_predict))
    
    # Definindo função objetivo da função de optimização (loss function):
    def objetivo(W):
        W1 = W[0:x.shape[0]*n].reshape(n,x.shape[0])
        b1 = W[x.shape[0]*n:x.shape[0]*n+n].reshape(n,1)
        W2 = W[x.shape[0]*n+n:-1].reshape(1,n)
        b2 = W[-1].reshape(1,1)
        Z1 = W1.dot(x) + b1
        A1 = sigmoid(Z1)
        Z2 = W2.dot(A1) + b2
        A2 = sigmoid(Z2)[0]
        loss = mse(y, A2)
        return math.sqrt(loss)

    def restricao1(W):
        W1 = W[0:x.shape[0]*n].reshape(n,x.shape[0])
        b1 = W[x.shape[0]*n:x.shape[0]*n+n].reshape(n,1)
        W2 = W[x.shape[0]*n+n:-1].reshape(1,n)
        b2 = W[-1].reshape(1,1)

        # Definindo valor base de x (tomando primeira entrada do conjunto de treino como referência)
        x_base = []
        for k in range(x.shape[0]):
            x_base.append(x[k][0])
        x_base = np.asarray(x_base)
        x_base = x_base.reshape(x.shape[0],1)
        # Definindo valor base de y
        Z1 = W1.dot(x_base) + b1
        A1 = sigmoid(Z1)
        Z2 = W2.dot(A1) + b2
        A2 = sigmoid(Z2)
        y_base = A2[0]

        # Registrando ganho referente à variavel 1 
        x_var = x_base.copy()
        x_var[0] = x_var[0]+0.1
        Z1 = W1.dot(x_var) + b1
        A1 = sigmoid(Z1)
        Z2 = W2.dot(A1) + b2
        A2 = sigmoid(Z2)
        new_y = A2[0]
        k1 = (new_y - y_base) / (x_var[0] - x_base[0])
        return -k1[0] - 0.0000000001

    def restricao2(W):
        W1 = W[0:x.shape[0]*n].reshape(n,x.shape[0])
        b1 = W[x.shape[0]*n:x.shape[0]*n+n].reshape(n,1)
        W2 = W[x.shape[0]*n+n:-1].reshape(1,n)
        b2 = W[-1].reshape(1,1)

        # Definindo valor base de x (tomando primeira entrada do conjunto de treino como referência)
        x_base = []
        for k in range(x.shape[0]):
            x_base.append(x[k][0])
        x_base = np.asarray(x_base)
        x_base = x_base.reshape(x.shape[0],1)
        # Definindo valor base de y
        Z1 = W1.dot(x_base) + b1
        A1 = sigmoid(Z1)
        Z2 = W2.dot(A1) + b2
        A2 = sigmoid(Z2)
        y_base = A2[0]

        # Registrando ganho referente à variavel 2
        x_var = x_base.copy()
        x_var[1] = x_var[1]+0.1
        Z1 = W1.dot(x_var) + b1
        A1 = sigmoid(Z1)
        Z2 = W2.dot(A1) + b2
        A2 = sigmoid(Z2)
        new_y = A2[0]
        k2 = (new_y - y_base) / (x_var[1] - x_base[1])
        return k2[0] - 0.0000000001

    def restricao3(W):
        W1 = W[0:x.shape[0]*n].reshape(n,x.shape[0])
        b1 = W[x.shape[0]*n:x.shape[0]*n+n].reshape(n,1)
        W2 = W[x.shape[0]*n+n:-1].reshape(1,n)
        b2 = W[-1].reshape(1,1)

        # Definindo valor base de x (tomando primeira entrada do conjunto de treino como referência)
        x_base = []
        for k in range(x.shape[0]):
            x_base.append(x[k][0])
        x_base = np.asarray(x_base)
        x_base = x_base.reshape(x.shape[0],1)
        # Definindo valor base de y
        Z1 = W1.dot(x_base) + b1
        A1 = sigmoid(Z1)
        Z2 = W2.dot(A1) + b2
        A2 = sigmoid(Z2)
        y_base = A2[0]

       # Registrando ganho referente à variavel 3
        x_var = x_base.copy()
        x_var[2] = x_var[2]+0.1
        Z1 = W1.dot(x_var) + b1
        A1 = sigmoid(Z1)
        Z2 = W2.dot(A1) + b2
        A2 = sigmoid(Z2)
        new_y = A2[0]
        k3 = (new_y - y_base) / (x_var[2] - x_base[2])
        return k3[0] - 0.0000000001

    def restricao4(W):
        W1 = W[0:x.shape[0]*n].reshape(n,x.shape[0])
        b1 = W[x.shape[0]*n:x.shape[0]*n+n].reshape(n,1)
        W2 = W[x.shape[0]*n+n:-1].reshape(1,n)
        b2 = W[-1].reshape(1,1)

        # Definindo valor base de x (tomando primeira entrada do conjunto de treino como referência)
        x_base = []
        for k in range(x.shape[0]):
            x_base.append(x[k][0])
        x_base = np.asarray(x_base)
        x_base = x_base.reshape(x.shape[0],1)
        # Definindo valor base de y
        Z1 = W1.dot(x_base) + b1
        A1 = sigmoid(Z1)
        Z2 = W2.dot(A1) + b2
        A2 = sigmoid(Z2)
        y_base = A2[0]

        # Registrando ganho referente à variavel 4
        x_var = x_base.copy()
        x_var[3] = x_var[3]+0.1
        Z1 = W1.dot(x_var) + b1
        A1 = sigmoid(Z1)
        Z2 = W2.dot(A1) + b2
        A2 = sigmoid(Z2)
        new_y = A2[0]
        k4 = (new_y - y_base) / (x_var[3] - x_base[3])
        return k4[0] - 0.0000000001

    def restricao5(W):
        W1 = W[0:x.shape[0]*n].reshape(n,x.shape[0])
        b1 = W[x.shape[0]*n:x.shape[0]*n+n].reshape(n,1)
        W2 = W[x.shape[0]*n+n:-1].reshape(1,n)
        b2 = W[-1].reshape(1,1)

        # Definindo valor base de x (tomando primeira entrada do conjunto de treino como referência)
        x_base = []
        for k in range(x.shape[0]):
            x_base.append(x[k][0])
        x_base = np.asarray(x_base)
        x_base = x_base.reshape(x.shape[0],1)
        # Definindo valor base de y
        Z1 = W1.dot(x_base) + b1
        A1 = sigmoid(Z1)
        Z2 = W2.dot(A1) + b2
        A2 = sigmoid(Z2)
        y_base = A2[0]

        # Registrando ganho referente à variavel 5
        x_var = x_base.copy()
        x_var[4] = x_var[4]+0.1
        Z1 = W1.dot(x_var) + b1
        A1 = sigmoid(Z1)
        Z2 = W2.dot(A1) + b2
        A2 = sigmoid(Z2)
        new_y = A2[0]
        k5 = (new_y - y_base) / (x_var[4] - x_base[4])
        return k5[0] - 0.0000000001

    def restricao6(W):
        W1 = W[0:x.shape[0]*n].reshape(n,x.shape[0])
        b1 = W[x.shape[0]*n:x.shape[0]*n+n].reshape(n,1)
        W2 = W[x.shape[0]*n+n:-1].reshape(1,n)
        b2 = W[-1].reshape(1,1)

        # Definindo valor base de x (tomando primeira entrada do conjunto de treino como referência)
        x_base = []
        for k in range(x.shape[0]):
            x_base.append(x[k][0])
        x_base = np.asarray(x_base)
        x_base = x_base.reshape(x.shape[0],1)
        # Definindo valor base de y
        Z1 = W1.dot(x_base) + b1
        A1 = sigmoid(Z1)
        Z2 = W2.dot(A1) + b2
        A2 = sigmoid(Z2)
        y_base = A2[0]

        # Registrando ganho referente à variavel 6
        x_var = x_base.copy()
        x_var[5] = x_var[5]+0.1
        Z1 = W1.dot(x_var) + b1
        A1 = sigmoid(Z1)
        Z2 = W2.dot(A1) + b2
        A2 = sigmoid(Z2)
        new_y = A2[0]
        k6 = (new_y - y_base) / (x_var[5] - x_base[5])
        return k6[0] - 0.0000000001

    rest1 = {'type': 'ineq', 'fun': restricao1}
    rest2 = {'type': 'ineq', 'fun': restricao2}
    rest3 = {'type': 'ineq', 'fun': restricao3}
    rest4 = {'type': 'ineq', 'fun': restricao4}
    rest5 = {'type': 'ineq', 'fun': restricao5}
    rest6 = {'type': 'ineq', 'fun': restricao6}

    cons = ([rest1, rest2, rest3, rest4, rest5, rest6])
    
    #Inicializando loop para demais neurônios ocultos:
    for n in range(2, 251):
        
      #Sinalização, em tela, de início de rede com n neurônios ocultos:
        print("Iniciando rede com",n,"neurônios ocultos")

        #Definindo entrada e saída de treino:
        x = inputs[train].T
        y = targets[train].T

        # Inicializando novos pesos e utilizando pesos do treinamento anterior:
        W1_in = np.random.uniform(-math.sqrt(6/(x.shape[0] + n)), math.sqrt(6/(x.shape[0] + n)), (n, x.shape[0]))
        b1_in = np.zeros((n, 1))    
        W2_in = np.random.uniform(-math.sqrt(6/(n + 1)), math.sqrt(6/(n + 1)), (1, n))
        b2_in = np.zeros((1, 1))
        W1_in[:-1] = W1
        b1_in[:-1] = b1
        W2_in[0][:-1] = W2
        b2_in = b2

        # Definindo vetor de pesos iniciais da função de optimização
        W0 = np.concatenate((W1_in.flatten(), b1_in.flatten(), W2_in.flatten(), b2_in.flatten()))
        
        # Implementando otimização
        solucao = minimize(objetivo, W0, method = 'SLSQP', constraints = cons)

        # Atribuindo pesos obtidos após à otimização
        W1 = solucao['x'][0:x.shape[0]*n].reshape(n,x.shape[0])
        b1 = solucao['x'][x.shape[0]*n:x.shape[0]*n+n].reshape(n,1)
        W2 = solucao['x'][x.shape[0]*n+n:-1].reshape(1,n)
        b2 = solucao['x'][-1].reshape(1,1)

        #Registro e exibição de MSE de treino:
        Z1 = W1.dot(x) + b1
        A1 = sigmoid(Z1)
        Z2 = W2.dot(A1) + b2
        A2 = sigmoid(Z2)
        y_predict = A2[0]
        Modelo1_loss = mse(y, y_predict)
        mse_train.append(Modelo1_loss)
        rmse_train.append(np.sqrt(Modelo1_loss))

        #Registro e exibição de R2 de treino:
        r2_train.append(r2_score(y, y_predict))

        #Definindo entrada e saída de teste:
        x = inputs[test].T
        y = targets[test].T

        #Testando modelo e calculando MSE de teste:
        Z1 = W1.dot(x) + b1
        A1 = sigmoid(Z1)
        Z2 = W2.dot(A1) + b2
        A2 = sigmoid(Z2)
        y_predict = A2[0]
        Modelo1_loss = mse(y, y_predict)
        mse_test.append(Modelo1_loss)
        rmse_test.append(np.sqrt(Modelo1_loss))

        #Registro e exibição de R2 de teste:
        r2_test.append(r2_score(y, y_predict))
        
    #Registrando RMSE de treino e teste:
    rmse_train_opt.append(rmse_train[r2_test.index(max(r2_test))])    
    rmse_test_opt.append(rmse_test[r2_test.index(max(r2_test))])
    
    #Registrando R2 máximo de treino e teste:
    r2_train_opt.append(r2_train[r2_test.index(max(r2_test))])    
    r2_test_opt.append(max(r2_test))
    l=l+1

In [ ]:
#Finalizando e exibindo contagem de tempo de processamento:
fim=time.time()
print("Tempo de processamento:",fim-ini)

In [ ]:
#Criação e exibição de Gráfico do RMSE de treino (corresponde ao RMSE de treino para o número ótimo de neurônios ocultos)
#ao longo das k combinações:
plt.plot(np.arange(1,l,step=1),rmse_train_opt)
plt.ylabel('RMSE de treino')
plt.xlabel('Validação')
plt.legend(['Constrained WILCA-R'])
plt.xticks(np.arange(1, l, step=1))
plt.show()

#Exibindo em tela R2 de teste máximo médio de treino:
print("RMSE de treino médio:",np.mean(rmse_train_opt))
print("Desvio padrão:",np.std(rmse_train_opt))

In [ ]:
#Criação e exibição de Gráfico do RMSE de teste (corresponde ao RMSE de treino para o número ótimo de neurônios ocultos)
#ao longo das k combinações:
plt.plot(np.arange(1,l),rmse_test_opt)
plt.ylabel('RMSE de teste')
plt.xlabel('Validação')
plt.legend(['Constrained WILCA-R'])
plt.xticks(np.arange(1, l, step=1))
plt.show()

#Exibindo em tela R2 de teste máximo médio de treino:
print("RMSE de teste médio:",np.mean(rmse_test_opt))
print("Desvio padrão:",np.std(rmse_test_opt))

In [ ]:
#Criação e exibição de Gráfico do R2 de treino (corresponde ao R2 de treino para o número ótimo de neurônios ocultos)
#ao longo das k combinações:
plt.plot(np.arange(1,l),r2_train_opt)
plt.ylabel('R2 de treino')
plt.xlabel('Validação')
plt.legend(['Constrained WILCA-R'])
plt.xticks(np.arange(1, l, step=1))
plt.show()

#Exibindo em tela R2 de teste máximo médio de treino:
print("R2 de treino médio:",np.mean(r2_train_opt))
print("Desvio padrão:",np.std(r2_train_opt))

In [ ]:
#Criação e exibição de Gráfico do R2 máximo de teste ao longo dos experimentos:
plt.plot(np.arange(1,l),r2_test_opt)
plt.ylabel('R2 de teste')
plt.xlabel('Validação')
plt.legend(['Constrained WILCA-R'])
plt.xticks(np.arange(1, l, step=1))
plt.show()

#Exibindo em tela R2 de teste máximo médio de teste:
print("R2 de teste médio:",np.mean(r2_test_opt))
print("Desvio padrão:",np.std(r2_test_opt))

## MÉTODO 2 COM RESTRIÇÕES DE GANHOS
Modelo SFNN considerando os pesos iniciais e bias do neurônio oculto e do neurônio de saída obtidos a partir do método de Xavier, desconsiderando os pesos dos neurônios já aprendidos.

### Criando Método 2:

### Executando Método 2:

In [ ]:
#Inicializando contagem do tempo:
ini=time.time()

#Abrindo listas para RMSE e R2 (treino e teste):

rmse_train_opt = []
rmse_test_opt = []
r2_train_opt = []
r2_test_opt = []

#Inicializando loop de experimentos:
l=1
kf = KFold(5, True, 1)
for train, test in kf.split(inputs, targets):
    np.random.seed(l)
    print("Validação cruzada",l,)
    #Abrindo listas para RMSE, MSE e R2 (treino e teste):
    mse_train = []
    mse_test = []
    rmse_train = []
    rmse_test = []
    r2_train = []
    r2_test = []
    
    # Definindo função objetivo da função de optimização (loss function):
    def objetivo(W):
        W1 = W[0:x.shape[0]*n].reshape(n,x.shape[0])
        b1 = W[x.shape[0]*n:x.shape[0]*n+n].reshape(n,1)
        W2 = W[x.shape[0]*n+n:-1].reshape(1,n)
        b2 = W[-1].reshape(1,1)
        Z1 = W1.dot(x) + b1
        A1 = sigmoid(Z1)
        Z2 = W2.dot(A1) + b2
        A2 = sigmoid(Z2)[0]
        loss = mse(y, A2)
        return math.sqrt(loss)

    def restricao1(W):
        W1 = W[0:x.shape[0]*n].reshape(n,x.shape[0])
        b1 = W[x.shape[0]*n:x.shape[0]*n+n].reshape(n,1)
        W2 = W[x.shape[0]*n+n:-1].reshape(1,n)
        b2 = W[-1].reshape(1,1)

        # Definindo valor base de x (tomando primeira entrada do conjunto de treino como referência)
        x_base = []
        for k in range(x.shape[0]):
            x_base.append(x[k][0])
        x_base = np.asarray(x_base)
        x_base = x_base.reshape(x.shape[0],1)
        # Definindo valor base de y
        Z1 = W1.dot(x_base) + b1
        A1 = sigmoid(Z1)
        Z2 = W2.dot(A1) + b2
        A2 = sigmoid(Z2)
        y_base = A2[0]

        # Registrando ganho referente à variavel 1 
        x_var = x_base.copy()
        x_var[0] = x_var[0]+0.1
        Z1 = W1.dot(x_var) + b1
        A1 = sigmoid(Z1)
        Z2 = W2.dot(A1) + b2
        A2 = sigmoid(Z2)
        new_y = A2[0]
        k1 = (new_y - y_base) / (x_var[0] - x_base[0])
        return -k1[0] - 0.0000000001

    def restricao2(W):
        W1 = W[0:x.shape[0]*n].reshape(n,x.shape[0])
        b1 = W[x.shape[0]*n:x.shape[0]*n+n].reshape(n,1)
        W2 = W[x.shape[0]*n+n:-1].reshape(1,n)
        b2 = W[-1].reshape(1,1)

        # Definindo valor base de x (tomando primeira entrada do conjunto de treino como referência)
        x_base = []
        for k in range(x.shape[0]):
            x_base.append(x[k][0])
        x_base = np.asarray(x_base)
        x_base = x_base.reshape(x.shape[0],1)
        # Definindo valor base de y
        Z1 = W1.dot(x_base) + b1
        A1 = sigmoid(Z1)
        Z2 = W2.dot(A1) + b2
        A2 = sigmoid(Z2)
        y_base = A2[0]

        # Registrando ganho referente à variavel 2
        x_var = x_base.copy()
        x_var[1] = x_var[1]+0.1
        Z1 = W1.dot(x_var) + b1
        A1 = sigmoid(Z1)
        Z2 = W2.dot(A1) + b2
        A2 = sigmoid(Z2)
        new_y = A2[0]
        k2 = (new_y - y_base) / (x_var[1] - x_base[1])
        return k2[0] - 0.0000000001

    def restricao3(W):
        W1 = W[0:x.shape[0]*n].reshape(n,x.shape[0])
        b1 = W[x.shape[0]*n:x.shape[0]*n+n].reshape(n,1)
        W2 = W[x.shape[0]*n+n:-1].reshape(1,n)
        b2 = W[-1].reshape(1,1)

        # Definindo valor base de x (tomando primeira entrada do conjunto de treino como referência)
        x_base = []
        for k in range(x.shape[0]):
            x_base.append(x[k][0])
        x_base = np.asarray(x_base)
        x_base = x_base.reshape(x.shape[0],1)
        # Definindo valor base de y
        Z1 = W1.dot(x_base) + b1
        A1 = sigmoid(Z1)
        Z2 = W2.dot(A1) + b2
        A2 = sigmoid(Z2)
        y_base = A2[0]

       # Registrando ganho referente à variavel 3
        x_var = x_base.copy()
        x_var[2] = x_var[2]+0.1
        Z1 = W1.dot(x_var) + b1
        A1 = sigmoid(Z1)
        Z2 = W2.dot(A1) + b2
        A2 = sigmoid(Z2)
        new_y = A2[0]
        k3 = (new_y - y_base) / (x_var[2] - x_base[2])
        return k3[0] - 0.0000000001

    def restricao4(W):
        W1 = W[0:x.shape[0]*n].reshape(n,x.shape[0])
        b1 = W[x.shape[0]*n:x.shape[0]*n+n].reshape(n,1)
        W2 = W[x.shape[0]*n+n:-1].reshape(1,n)
        b2 = W[-1].reshape(1,1)

        # Definindo valor base de x (tomando primeira entrada do conjunto de treino como referência)
        x_base = []
        for k in range(x.shape[0]):
            x_base.append(x[k][0])
        x_base = np.asarray(x_base)
        x_base = x_base.reshape(x.shape[0],1)
        # Definindo valor base de y
        Z1 = W1.dot(x_base) + b1
        A1 = sigmoid(Z1)
        Z2 = W2.dot(A1) + b2
        A2 = sigmoid(Z2)
        y_base = A2[0]

        # Registrando ganho referente à variavel 4
        x_var = x_base.copy()
        x_var[3] = x_var[3]+0.1
        Z1 = W1.dot(x_var) + b1
        A1 = sigmoid(Z1)
        Z2 = W2.dot(A1) + b2
        A2 = sigmoid(Z2)
        new_y = A2[0]
        k4 = (new_y - y_base) / (x_var[3] - x_base[3])
        return k4[0] - 0.0000000001

    def restricao5(W):
        W1 = W[0:x.shape[0]*n].reshape(n,x.shape[0])
        b1 = W[x.shape[0]*n:x.shape[0]*n+n].reshape(n,1)
        W2 = W[x.shape[0]*n+n:-1].reshape(1,n)
        b2 = W[-1].reshape(1,1)

        # Definindo valor base de x (tomando primeira entrada do conjunto de treino como referência)
        x_base = []
        for k in range(x.shape[0]):
            x_base.append(x[k][0])
        x_base = np.asarray(x_base)
        x_base = x_base.reshape(x.shape[0],1)
        # Definindo valor base de y
        Z1 = W1.dot(x_base) + b1
        A1 = sigmoid(Z1)
        Z2 = W2.dot(A1) + b2
        A2 = sigmoid(Z2)
        y_base = A2[0]

        # Registrando ganho referente à variavel 5
        x_var = x_base.copy()
        x_var[4] = x_var[4]+0.1
        Z1 = W1.dot(x_var) + b1
        A1 = sigmoid(Z1)
        Z2 = W2.dot(A1) + b2
        A2 = sigmoid(Z2)
        new_y = A2[0]
        k5 = (new_y - y_base) / (x_var[4] - x_base[4])
        return k5[0] - 0.0000000001

    def restricao6(W):
        W1 = W[0:x.shape[0]*n].reshape(n,x.shape[0])
        b1 = W[x.shape[0]*n:x.shape[0]*n+n].reshape(n,1)
        W2 = W[x.shape[0]*n+n:-1].reshape(1,n)
        b2 = W[-1].reshape(1,1)

        # Definindo valor base de x (tomando primeira entrada do conjunto de treino como referência)
        x_base = []
        for k in range(x.shape[0]):
            x_base.append(x[k][0])
        x_base = np.asarray(x_base)
        x_base = x_base.reshape(x.shape[0],1)
        # Definindo valor base de y
        Z1 = W1.dot(x_base) + b1
        A1 = sigmoid(Z1)
        Z2 = W2.dot(A1) + b2
        A2 = sigmoid(Z2)
        y_base = A2[0]

        # Registrando ganho referente à variavel 6
        x_var = x_base.copy()
        x_var[5] = x_var[5]+0.1
        Z1 = W1.dot(x_var) + b1
        A1 = sigmoid(Z1)
        Z2 = W2.dot(A1) + b2
        A2 = sigmoid(Z2)
        new_y = A2[0]
        k6 = (new_y - y_base) / (x_var[5] - x_base[5])
        return k6[0] - 0.0000000001

    rest1 = {'type': 'ineq', 'fun': restricao1}
    rest2 = {'type': 'ineq', 'fun': restricao2}
    rest3 = {'type': 'ineq', 'fun': restricao3}
    rest4 = {'type': 'ineq', 'fun': restricao4}
    rest5 = {'type': 'ineq', 'fun': restricao5}
    rest6 = {'type': 'ineq', 'fun': restricao6}

    cons = ([rest1, rest2, rest3, rest4, rest5, rest6])
    
    #Inicializando loop para neurônios ocultos:
    for n in range(1, 251):
        
      #Sinalização, em tela, de início de rede com n neurônios ocultos:
        print("Iniciando rede com",n,"neurônios ocultos")

        #Definindo entrada e saída de treino:
        x = inputs[train].T
        y = targets[train].T

        # Inicializando novos pesos:
        W1_in = np.random.uniform(-math.sqrt(6/(x.shape[0] + n)), math.sqrt(6/(x.shape[0] + n)), (n, x.shape[0]))
        b1_in = np.zeros((n, 1))    
        W2_in = np.random.uniform(-math.sqrt(6/(n + 1)), math.sqrt(6/(n + 1)), (1, n))
        b2_in = np.zeros((1, 1))

        # Definindo vetor de pesos iniciais da função de optimização
        W0 = np.concatenate((W1_in.flatten(), b1_in.flatten(), W2_in.flatten(), b2_in.flatten()))
        
        # Implementando otimização
        solucao = minimize(objetivo, W0, method = 'SLSQP', constraints = cons)

        # Atribuindo pesos obtidos após à otimização
        W1 = solucao['x'][0:x.shape[0]*n].reshape(n,x.shape[0])
        b1 = solucao['x'][x.shape[0]*n:x.shape[0]*n+n].reshape(n,1)
        W2 = solucao['x'][x.shape[0]*n+n:-1].reshape(1,n)
        b2 = solucao['x'][-1].reshape(1,1)

        #Registro e exibição de MSE de treino:
        Z1 = W1.dot(x) + b1
        A1 = sigmoid(Z1)
        Z2 = W2.dot(A1) + b2
        A2 = sigmoid(Z2)
        y_predict = A2[0]
        Modelo2_loss = mse(y, y_predict)
        mse_train.append(Modelo2_loss)
        rmse_train.append(np.sqrt(Modelo2_loss))

        #Registro e exibição de R2 de treino:
        r2_train.append(r2_score(y, y_predict))

        #Definindo entrada e saída de teste:
        x = inputs[test].T
        y = targets[test].T

        #Testando modelo e calculando MSE de teste:
        Z1 = W1.dot(x) + b1
        A1 = sigmoid(Z1)
        Z2 = W2.dot(A1) + b2
        A2 = sigmoid(Z2)
        y_predict = A2[0]
        Modelo2_loss = mse(y, y_predict)
        mse_test.append(Modelo2_loss)
        rmse_test.append(np.sqrt(Modelo2_loss))

        #Registro e exibição de R2 de teste:
        r2_test.append(r2_score(y, y_predict))
        
    #Registrando RMSE de treino e teste:
    rmse_train_opt.append(rmse_train[r2_test.index(max(r2_test))])    
    rmse_test_opt.append(rmse_test[r2_test.index(max(r2_test))])
    
    #Registrando R2 máximo de treino e teste:
    r2_train_opt.append(r2_train[r2_test.index(max(r2_test))])    
    r2_test_opt.append(max(r2_test))
    l=l+1

In [ ]:
#Finalizando e exibindo contagem de tempo de processamento:
fim=time.time()
print("Tempo de processamento:",fim-ini)

In [ ]:
#Criação e exibição de Gráfico do RMSE de treino (corresponde ao RMSE de treino para o número ótimo de neurônios ocultos)
#ao longo das k combinações:
plt.plot(np.arange(1,l,step=1),rmse_train_opt)
plt.ylabel('RMSE de treino')
plt.xlabel('Validação')
plt.legend(['RIXM'])
plt.xticks(np.arange(1, l, step=1))
plt.show()

#Exibindo em tela R2 de teste máximo médio de treino:
print("RMSE de treino médio:",np.mean(rmse_train_opt))
print("Desvio padrão:",np.std(rmse_train_opt))

In [ ]:
#Criação e exibição de Gráfico do RMSE de teste (corresponde ao RMSE de teste para o número ótimo de neurônios ocultos)
#ao longo das k combinações:
plt.plot(np.arange(1,l),rmse_test_opt)
plt.ylabel('RMSE de teste')
plt.xlabel('Validação')
plt.legend(['RIXM'])
plt.xticks(np.arange(1, l, step=1))
plt.show()

#Exibindo em tela R2 de teste máximo médio de treino:
print("RMSE de teste médio:",np.mean(rmse_test_opt))
print("Desvio padrão:",np.std(rmse_test_opt))

In [ ]:
#Criação e exibição de Gráfico do R2 de treino (corresponde ao R2 de treino para o número ótimo de neurônios ocultos)
#ao longo das k combinações:
plt.plot(np.arange(1,l),r2_train_opt)
plt.ylabel('R2 de treino')
plt.xlabel('Validação')
plt.legend(['RIXM'])
plt.xticks(np.arange(1, l, step=1))
plt.show()

#Exibindo em tela R2 de teste máximo médio de treino:
print("R2 de treino médio:",np.mean(r2_train_opt))
print("Desvio padrão:",np.std(r2_train_opt))

In [ ]:
#Criação e exibição de Gráfico do R2 máximo de teste ao longo dos experimentos:
plt.plot(np.arange(1,l),r2_test_opt)
plt.ylabel('R2 de teste')
plt.xlabel('Validação')
plt.legend(['RIXM'])
plt.xticks(np.arange(1, l, step=1))
plt.show()

#Exibindo em tela R2 de teste máximo médio de teste:
print("R2 de teste médio:",np.mean(r2_test_opt))
print("Desvio padrão:",np.std(r2_test_opt))

### MÉTODO 3
Modelo ELM considerando os pesos iniciais e bias do neurônio oculto e do neurônio de saída obtidos a partir do método de inicialização randômica clássica, desconsiderando os pesos já aprendidos.

### Definindo função de ativação ReLU:

In [ ]:
def relu(x):
    return np.maximum(x, 0, x)

### Criando Método 3:

In [ ]:
def hidden_nodes(X):
    G = np.dot(X, input_weights)
    G = G + biases
    H = relu(G)
    return H
def predict_test(X):
    out = hidden_nodes(X)
    out = np.dot(out, output_weights)
    return out

### Executando Método 3:

In [ ]:
#Inicializando contagem do tempo:
ini=time.time()

#Abrindo listas para RMSE e R2 ótimos (treino e teste):
rmse_train_opt = []
rmse_test_opt = []
r2_train_opt = []
r2_test_opt = []

#Inicializando loop de experimentos:
l=1
kf = KFold(5, True, 1)
for train, test in kf.split(inputs, targets):
    np.random.seed(l)
    print("Validação cruzada",l,)
    #Abrindo listas para RMSE e R2 (treino e teste):
    rmse_train = []
    rmse_test = []
    r2_train = []
    r2_test = []

    #Inicializando loop para neurônios ocultos:
    for n in range(1,251):
        
        #Treino:
        input_weights = np.random.uniform(-1, 1, (input_size,n))
        biases = np.zeros(n)
        print("Iniciando rede com",n,"neurônios ocultos")
        out = hidden_nodes(inputs[train])
        output_weights = np.dot(pinv2(out), targets[train])
        out = np.dot(out, output_weights)
        actual = targets[train]
        r2_train.append(r2_score(actual, out))
        rmse_train.append(np.sqrt(mse(actual,out)))

        #Teste:
        test_predict = predict_test(inputs[test])
        actual = targets[test]
        r2_test.append(r2_score(actual, test_predict))
        rmse_test.append(np.sqrt(mse(actual,test_predict)))
        
    #Registrando RMSE de treino e teste:
    rmse_train_opt.append(rmse_train[r2_test.index(max(r2_test))])    
    rmse_test_opt.append(rmse_test[r2_test.index(max(r2_test))])
    #Registrando R2 máximo de treino e teste:
    r2_train_opt.append(r2_train[r2_test.index(max(r2_test))])    
    r2_test_opt.append(max(r2_test))
    l=l+1

In [ ]:
#Finalizando e exibindo contagem de tempo de processamento:
fim=time.time()
print("Tempo de processamento:",fim-ini)

In [ ]:
#Criação e exibição de Gráfico do RMSE de treino (corresponde ao RMSE de treino para o número ótimo de neurônios ocultos)
#ao longo das k combinações:
plt.plot(np.arange(1,l,step=1),rmse_train_opt)
plt.ylabel('RMSE de treino')
plt.xlabel('Validação')
plt.legend(['ELM'])
plt.xticks(np.arange(1, l, step=1))
plt.show()

#Exibindo em tela R2 de teste máximo médio de treino:
print("RMSE de treino médio:",np.mean(rmse_train_opt))
print("Desvio padrão:",np.std(rmse_train_opt))

In [ ]:
#Criação e exibição de Gráfico do RMSE de teste (corresponde ao RMSE de teste para o número ótimo de neurônios ocultos)
#ao longo das k combinações:
plt.plot(np.arange(1,l),rmse_test_opt)
plt.ylabel('RMSE de teste')
plt.xlabel('Validação')
plt.legend(['ELM'])
plt.xticks(np.arange(1, l, step=1))
plt.show()

#Exibindo em tela R2 de teste máximo médio de treino:
print("RMSE de teste médio:",np.mean(rmse_test_opt))
print("Desvio padrão:",np.std(rmse_test_opt))

In [ ]:
#Criação e exibição de Gráfico do R2 de treino (corresponde ao R2 de treino para o número ótimo de neurônios ocultos)
#ao longo das k combinações:
plt.plot(np.arange(1,l),r2_train_opt)
plt.ylabel('R2 de treino')
plt.xlabel('Validação')
plt.legend(['ELM'])
plt.xticks(np.arange(1, l, step=1))
plt.show()

#Exibindo em tela R2 de teste máximo médio de treino:
print("R2 de treino médio:",np.mean(r2_train_opt))
print("Desvio padrão:",np.std(r2_train_opt))

In [ ]:
#Criação e exibição de Gráfico do R2 máximo de teste ao longo dos experimentos:
plt.plot(np.arange(1,l),r2_test_opt)
plt.ylabel('R2 de teste')
plt.xlabel('Validação')
plt.legend(['ELM'])
plt.xticks(np.arange(1, l, step=1))
plt.show()

#Exibindo em tela R2 de teste máximo médio de teste:
print("R2 de teste médio:",np.mean(r2_test_opt))
print("Desvio padrão:",np.std(r2_test_opt))

### FIM DO CÓDIGO